# 2.1 MCP — Model Context Protocol

## What you will learn in this notebook

- What **MCP (Model Context Protocol)** is and why it exists
- The three things an MCP server can expose: **Tools**, **Resources**, and **Prompts**
- How to connect to a **local MCP server** (runs as a subprocess on your machine)
- How to connect to an **online MCP server** (runs as a remote process via `uvx`)
- How to use `await` / `async` when working with MCP in notebooks

---

### What is MCP?

**MCP (Model Context Protocol)** is an open standard created by Anthropic that defines a universal way for AI agents to connect to external tools, data sources, and services — without each one needing custom integration code.

Think of it like **USB for AI agents**:
- Before USB, every peripheral needed its own proprietary connector and driver
- USB created one universal standard — plug anything in, it just works
- Before MCP, every tool integration needed custom glue code per agent framework
- MCP creates one universal standard — connect any agent to any server

```
WITHOUT MCP:                    WITH MCP:

Agent ──── custom code ──── Slack      Agent ──── MCP ──── Slack MCP Server
Agent ──── custom code ──── GitHub     Agent ──── MCP ──── GitHub MCP Server
Agent ──── custom code ──── DB         Agent ──── MCP ──── DB MCP Server
```

### What can an MCP server expose?

| Capability | What it is | Example |
|---|---|---|
| **Tools** | Functions the agent can call (same as `@tool` but served remotely) | `search_web(query)` |
| **Resources** | Read-only data the agent can access | A GitHub README file |
| **Prompts** | Pre-written system prompts the server defines | A domain-specific persona |

### Two transport types

| Transport | How it works | Use case |
|---|---|---|
| **stdio** | Agent spawns the server as a subprocess, communicates via stdin/stdout | Local servers (your own code, `uvx` packages) |
| **streamable_http** | Agent connects to a remote URL over HTTP | Hosted/cloud MCP servers |

### Why `await`?

MCP communication is **asynchronous** — connecting to a server, fetching tools, and calling them all involve network/process I/O that shouldn't block the main thread. In Jupyter notebooks, `await` works directly in cells (no need to wrap in `asyncio.run()`). In plain `.py` scripts, you'd use `asyncio.run()` or an async framework.

In [3]:
# ============================================================
# CELL 1: Load Environment Variables
# ============================================================
# Loads OPENAI_API_KEY and TAVILY_API_KEY.
# The MCP server (2.1_mcp_server.py) also calls load_dotenv()
# independently when it starts up as a subprocess.

from dotenv import load_dotenv

load_dotenv()

True

In [4]:
# ============================================================
# CELL 2: Windows Compatibility Fix
# ============================================================
# MCP servers communicate via subprocesses (stdio transport).
# Windows requires specific asyncio setup to support subprocesses:
#
# Fix 1: WindowsProactorEventLoopPolicy
#   The default SelectorEventLoop on Windows does NOT support
#   subprocesses. ProactorEventLoop does. We switch to it here.
#   This is only needed on Windows — Linux/Mac work out of the box.
#
# Fix 2: sys.stderr redirect
#   Jupyter's kernel replaces sys.stderr with an object that doesn't
#   have a fileno() method (needed when forking subprocesses).
#   We restore sys.__stderr__ (the real stderr) so subprocess
#   launching doesn't raise a fileno() AttributeError.
#
# This cell is a no-op on Mac/Linux — safe to leave in place.

import sys
import asyncio

if sys.platform == "win32":
    # Switch to ProactorEventLoop which supports subprocesses on Windows
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # Restore real stderr so subprocess forking works inside Jupyter
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

---
## Section 1 — Local MCP Server

A **local MCP server** runs as a subprocess on your own machine. LangChain spawns it, talks to it via stdin/stdout (stdio transport), and kills it when done.

Our local server (`resources/2.1_mcp_server.py`) exposes:
- **Tool**: `search_web` — Tavily-powered web search
- **Resource**: GitHub README file for langchain-mcp-adapters
- **Prompt**: A domain-specific LangChain expert persona

```
Jupyter Kernel
     │
     │  (spawns subprocess via stdio transport)
     ▼
2.1_mcp_server.py
     │  exposes: tools, resources, prompts
     │
     ├── Tool: search_web(query)
     ├── Resource: github://...README.md
     └── Prompt: LangChain expert persona
```

In [5]:
# ============================================================
# CELL 3: Connect to the Local MCP Server
# ============================================================
# MultiServerMCPClient manages connections to one or more MCP servers.
# It's a dict where each key is a server nickname and the value
# is a config dict describing how to connect.
#
# Config fields for stdio transport:
#   "transport": "stdio"           → communicate via stdin/stdout
#   "command":   "python"          → the executable to run
#   "args":      ["path/to/server.py"] → arguments passed to the command
#
# When this cell runs, LangChain launches:
#   python resources/2.1_mcp_server.py
# as a background subprocess. The MCP protocol handshake happens
# automatically — we never see it.
#
# MultiServerMCPClient can manage MULTIPLE servers simultaneously:
#   {"server_a": {...}, "server_b": {...}}
# Tools from all servers are merged into one list for the agent.

from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {              # Nickname for this server
            "transport": "streamable_http",        # HTTP transport (not stdio)
            "url": "http://127.0.0.1:8000/mcp"
        }
    }
)

In [6]:
# ============================================================
# CELL 4: Fetch Tools, Resources, and Prompts from the Server
# ============================================================
# MCP exposes three types of capabilities — we fetch each separately:
#
# client.get_tools()
#   Returns all @mcp.tool() functions as LangChain tool objects.
#   These are ready to pass directly to create_agent(tools=[...]).
#   No server name needed — tools from all connected servers are merged.
#
# client.get_resources("local_server")
#   Returns read-only data objects (@mcp.resource() decorators).
#   Server name IS needed because resources are per-server.
#   Resources are like files/documents the agent can read.
#
# client.get_prompt("local_server", "prompt")
#   Returns a specific named prompt template (@mcp.prompt() decorator).
#   We take [0].content to get the raw string from the first message.
#   This becomes our agent's system_prompt — defined by the SERVER,
#   not by us. This is powerful: the server author controls the persona.
#
# All three calls use `await` because they involve IPC with the subprocess.

tools = await client.get_tools()                           # All tools from all servers
resources = await client.get_resources("local_server")     # Read-only data from local_server
prompt = await client.get_prompt("local_server", "prompt") # Named prompt template
prompt = prompt[0].content                                 # Extract the raw string

In [7]:
tools

[StructuredTool(name='search_web', description='Search the web for information', args_schema={'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'search_webArguments', 'type': 'object'}, handle_tool_error=<function _handle_mcp_tool_error at 0x000002649A278680>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x000002649A32B600>)]

In [8]:
# ============================================================
# CELL 5: Create the Agent Using MCP-Sourced Tools and Prompt
# ============================================================
# Notice what's different here compared to Module 1:
#
#   tools=tools       → came from the MCP server, not defined locally
#   system_prompt=prompt → came from the MCP server, not written by us
#
# This is the key power of MCP: the server author defines BOTH
# the capabilities AND the persona. As a consumer of the server,
# you get a fully configured, domain-expert agent with zero tool
# or prompt code on your side.
#
# In a production scenario:
#   - A Slack MCP server would give you Slack tools + Slack-aware prompt
#   - A GitHub MCP server would give you GitHub tools + code-review prompt
#   - You consume them all the same way

from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=tools,        # Tools sourced from MCP server
    system_prompt=prompt # Prompt sourced from MCP server
)

In [9]:
# ============================================================
# CELL 6: Invoke the Agent Asynchronously
# ============================================================
# We use agent.ainvoke() instead of agent.invoke().
#
# Why ainvoke() (async invoke) instead of invoke() (sync)?
#   MCP tool calls go through async subprocess communication.
#   Using the async version avoids blocking the event loop
#   and is required when working inside an async context
#   (which Jupyter notebooks provide by default).
#
# Everything else is identical to Module 1:
#   - dict with 'messages' key
#   - config with thread_id for memory
#   - response['messages'][-1].content for the final answer

from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [10]:
# ============================================================
# CELL 7: Inspect the Full Response
# ============================================================
# The message trace shows the agent using the MCP-sourced tools.
# Look for messages[1].tool_calls to see which MCP tool was called
# and with what query — the model chose to search_web because the
# system prompt instructed it to use available tools.

from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='bdd09e9a-673a-4ab0-90e3-15abfcaf4d29'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_web', 'arguments': '{"query": "langchain-mcp-adapters library"}'}, '__gemini_function_call_thought_signatures__': {'1f0b9549-f6f5-4e5a-b29b-19964b21343f': 'Cq8BARFNMg88KQyT26Njp2tfVdfaFbt4Ai79TU74SQvC/RBPEz2kneDGhL3gTPC30F6KYbl0W4U7uXU1VNHRebIp8YkIvWEwaoVr8sLgfwRO6yLJMwMqYHuhzLwlqTmiKwOtQlz+bJEIAgF/Xkjtd9WZx1gaNj0dxyDcNc4MBrqTStDfggTj7HkxYyfj0mIPtF0cYj8azZN3KS2pSQlwf9WgyE84p6B3hirhIBoYi8hhgQ=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f3d6b-4c93-75a1-8577-97c7c206f351-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters library'}, 'id': '1f0b9549-f6f5-4e5a-b29b-19964b21343f', 'type': 'tool

---
## Section 2 — Online MCP Server

An **online MCP server** is a pre-built, community-published server that you run via `uvx` — Python's zero-install tool runner. You don't write any server code; you just reference it by package name.

`uvx mcp-server-time` downloads and runs the `mcp-server-time` package from PyPI in an isolated environment. It exposes a tool for getting the current time in any timezone.

### The MCP ecosystem

There are hundreds of community MCP servers available:
- `mcp-server-time` — current time in any timezone
- `mcp-server-git` — read/write Git repositories
- `mcp-server-postgres` — query a PostgreSQL database
- `mcp-server-slack` — read/post Slack messages
- `mcp-server-filesystem` — read/write local files

Browse them at: https://github.com/modelcontextprotocol/servers

In [ ]:
# ============================================================
# CELL 8: Connect to the mcp-server-time Online Package
# ============================================================
# Still using stdio transport — but instead of running a local .py file,
# we use "uvx" as the command to run a published PyPI package.
#
# uvx is like npx for Python — it downloads the package, creates
# an isolated virtual environment, and runs it. No pip install needed.
#
# Config fields:
#   "command": "uvx"                    → use uvx runner
#   "args": ["mcp-server-time",         → package name on PyPI
#             "--local-timezone=..."]   → argument passed to the server
#
# The --local-timezone argument tells the time server what timezone
# to treat as "local" when answering "what time is it here?" questions.
#
# After get_tools(), `tools` contains whatever the time server exposes —
# typically: get_current_time(timezone), convert_time(from, to, time)

client = MultiServerMCPClient(
    {
        "time": {                              # Nickname for the time server
            "transport": "stdio",
            "command": "uvx",                  # Run via uvx (zero-install)
            "args": [
                "mcp-server-time",             # Package name on PyPI
                "--local-timezone=America/New_York"  # Server config flag
            ]
        }
    }
)

tools = await client.get_tools()  # Fetch tools from the time server

In [ ]:
# ============================================================
# CELL 9: Create an Agent with the Time Tool
# ============================================================
# No system prompt needed here — the agent will use the time
# tool naturally when asked a time-related question.

agent = create_agent(
    model="gpt-5-nano",
    tools=tools,   # Time tools from the MCP server
)

In [ ]:
# ============================================================
# CELL 10: Ask "What time is it?" — A Question LLMs Can't Answer Alone
# ============================================================
# Without a tool, the model cannot know the current time —
# its training data has no real-time clock.
#
# With the time MCP server, the agent calls get_current_time()
# and gets the exact current time in the configured timezone.
#
# This demonstrates a pure capability gap that MCP fills:
# not web search (which the model could partially approximate),
# but a fundamentally real-time fact that requires a live tool.

question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

---
## Summary

| Concept | Key takeaway |
|---|---|
| **MCP** | Universal protocol — connect any agent to any tool server without custom glue code |
| **Tools** | Callable functions exposed by the server — fetched with `get_tools()` |
| **Resources** | Read-only data (files, docs) — fetched with `get_resources(server_name)` |
| **Prompts** | System prompts defined by the server — fetched with `get_prompt(server, name)` |
| **stdio transport** | Server runs as a local subprocess — your code or a `uvx` package |
| **streamable_http** | Server runs remotely — connect to a URL (see travel agent notebook) |
| **`ainvoke()`** | Async version of `invoke()` — required when tools involve async I/O |
| **`uvx`** | Runs any PyPI MCP package without installing it — like npx for Python |

### Next steps
- **2.1 Travel Agent** — connect to a hosted HTTP MCP server (Kiwi.com flight search)
- **2.2 Runtime Context** — pass per-request user data to tools without storing it in state
- Browse available MCP servers: https://github.com/modelcontextprotocol/servers